<a href="https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2010%20-%20Unsupervised%20Learning%20and%20Clustering/Unsupervised_Learning_and_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IS834 — Session 10**

Work through this notebook in Colab. The lecture cells are filled in and ready to run; the
**Case** and the **Exercises** are yours to complete — write your code in the `# Your code here` cells and your
reasoning in the `*Provide your answer here*` cells.

Every dataset is read straight from the course repository, so there is nothing to upload or install.

*Adapted from Brock Tibert's IS834 Fall 2025 notebook `10_ML2_Supervised_and_Unsupervised.ipynb`. Edited for Fall 2026 by Mohannad Elhamod.*

# Session 10 — Unsupervised learning: clustering

**Learning goals**

1. What KMeans clustering does, and what it needs from your data
2. Standardising features, and why clustering is close to meaningless without it
3. Choosing *k* with the silhouette score
4. Judging when a clustering answers a business question and when it doesn't


## Setup

The clustering work below runs on the **bank** dataset — the same two files you joined in sessions 8 and 9. Run these cells first to rebuild it.


In [ ]:
# Everything this notebook needs, imported once at the top.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Show wide tables without pandas truncating them to a handful of columns.
pd.set_option('display.max_columns', 100)

# Every data file this notebook uses is read straight from the course repository.
# There is nothing to upload: DATA is the folder those files live in.
DATA = "https://raw.githubusercontent.com/elhamod/IS834_Fall_2026/main/data/"

In [ ]:
# Read the two bank files and join them back into one table on the customer id.
customers = pd.read_csv(DATA + "bank-customers.csv")
marketing = pd.read_csv(DATA + "bank-marketing.csv")

bank = pd.merge(customers, marketing, on="id", how="left")
print(bank.shape)
bank.info()


### Bank marketing dataset overview

| **Attribute** | **Description** | **Type / Values** |
|----------------|-----------------|-------------------|
| **id** | The bank customer's id | String |
| **age** | Age of the client | Numeric |
| **job** | Type of job | Categorical: *admin.*, *unknown*, *unemployed*, *management*, *housemaid*, *entrepreneur*, *student*, *blue-collar*, *self-employed*, *retired*, *technician*, *services* |
| **marital** | Marital status | Categorical: *married*, *divorced*, *single* |
| **education** | Level of education | Categorical: *unknown*, *primary*, *secondary*, *tertiary* |
| **default** | Has credit in default? | Binary: *yes*, *no* |
| **balance** | Average yearly balance (in euros) | Numeric |
| **housing** | Has housing loan? | Binary: *yes*, *no* |
| **loan** | Has personal loan? | Binary: *yes*, *no* |
| **contact** | Contact communication type | Categorical: *unknown*, *telephone*, *cellular* |
| **day** | Last contact day of the month | Numeric |
| **month** | Last contact month of year | Categorical: *jan*, *feb*, ..., *dec* |
| **duration** | Last contact duration (in seconds) | Numeric |
| **campaign** | Number of contacts performed during this campaign (includes last contact) | Numeric |
| **pdays** | Days passed since client was last contacted (-1 = never contacted) | Numeric |
| **previous** | Number of contacts before this campaign | Numeric |
| **poutcome** | Outcome of previous marketing campaign | Categorical: *unknown*, *other*, *failure*, *success* |
| **y** | Client subscribed a term deposit? | Binary: *yes*, *no* |


## Clustering with KMeans

### What KMeans needs

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Cluster-2.svg/1200px-Cluster-2.svg.png"  height="300" width="420">

![](https://cdn-images-1.medium.com/max/1600/1*tWaaZX75oumVwBMcKN-eHA.png)

**Inputs.**

- a DataFrame of **numeric** columns, used to measure how similar two rows are
- the number of clusters we want, commonly called `k`

**Output.**

- a cluster number for each row of the DataFrame, from 0 to `k` − 1

KMeans measures the distance between rows, and you cannot take the distance between "married" and "single". So it only works on numeric columns.

In [ ]:
# A few random rows, so we can see what we are clustering.
bank.sample(3, random_state=595)


In [ ]:
# KMeans needs numbers, so keep only the numeric columns.
bank_numeric = bank.select_dtypes("number")
bank_numeric.head(3)

In [ ]:
# Summary statistics -- compare the scales of the columns.
bank_numeric.describe().round(1)

**Question 1.** Compare the standard deviation of `balance` with the standard deviation of `campaign`. What is about to go wrong?


**Answer**

*Provide your answer here*

In [ ]:
# seaborn's pairplot shows every pair of variables at once.
# 500 rows is a sample -- plotting all 45,211 points would be slow and unreadable.
sns.pairplot(bank_numeric.sample(500, random_state=595))
plt.show()


### First, the trap: clustering the raw numbers

Before we standardise anything, let us run KMeans exactly as it comes and look honestly at what it produces.


In [ ]:
# Cluster the RAW columns -- deliberately skipping the standardisation step.
raw_kmeans = KMeans(n_clusters=3, random_state=595, n_init=10)
bank["cluster_raw"] = raw_kmeans.fit_predict(bank_numeric)

# The average of each variable inside each cluster, plus how many customers landed there.
raw_profile = bank.groupby("cluster_raw")[bank_numeric.columns.tolist()].mean().round(1)
raw_profile["n_customers"] = bank["cluster_raw"].value_counts().sort_index()
raw_profile


**Question 2.** Read the table above. What did the algorithm actually group people by?


**Answer**

*Provide your answer here*

### The fix: standardise first

A **z-score** rewrites each value as "how many standard deviations from the mean of its own column". After the transformation every column has mean 0 and standard deviation 1, so no column can dominate by virtue of its units.


![](https://agitator.thedonorvoice.com/wp-content/uploads/2022/08/z-score-1024x756.png)


In [ ]:
# StandardScaler converts every column to a z-score: (value - column mean) / column std.
scaler = StandardScaler()
bank_scaled = scaler.fit_transform(bank_numeric)

# fit_transform hands back a plain array, so wrap it in a DataFrame to inspect it.
# Every mean should now be 0 and every standard deviation 1.
bank_scaled_df = pd.DataFrame(bank_scaled, columns=bank_numeric.columns)
bank_scaled_df.describe().round(3)

In [ ]:
# Cluster the standardised data. k = 3 is a starting guess; we test it below.
kmeans = KMeans(n_clusters=3, random_state=595, n_init=10)
clusters = kmeans.fit_predict(bank_scaled)

# One cluster number per row, counting from zero.
print(clusters[:5])

In [ ]:
# Attach the assignment back onto the original dataframe so we can describe the groups.
bank["k3"] = clusters

sns.countplot(data=bank, x="k3")
plt.title("How many customers in each cluster")
plt.show()


### Describing the clusters

A cluster number is not an insight. To say anything useful you have to look at what is *typical* inside each group.

**Exercise 1.** Calculate the average of every numeric variable within each cluster, on both the original scale and the standardised scale, and describe the three groups in plain English.


In [ ]:
# Your code here -- the average of every numeric variable in each cluster, in the original units

In [ ]:
# Your code here -- the same averages on the standardised scale

**Answer**

*Provide your answer here*

### Choosing *k*

![](https://miro.medium.com/v2/resize:fit:720/1*ftfk97tHNG9pF25BRD8GsA.png)

The **silhouette score** is a popular way to judge a clustering. It ranges from +1 (ideal) to −1 (very bad).

- Every row gets a score: how close it is to the rows in its own cluster, balanced against how far it is from the nearest other cluster.
- The row scores are averaged into one number for the whole solution.
- In business terms: each cluster should be compact, and clearly different from the others.

To choose *k*, fit a model for each candidate value, score it, and plot the scores.

In [ ]:
# Fit and score a model for each k from 2 to 9.
# A 2,000-row sample keeps this fast enough to run in class; the pattern holds on the full data.
sample_scaled = pd.DataFrame(bank_scaled, columns=bank_numeric.columns).sample(2000, random_state=595)

scores = []
for k in range(2, 10):
    trial = KMeans(n_clusters=k, random_state=595, n_init=10)
    labels = trial.fit_predict(sample_scaled)
    scores.append(silhouette_score(sample_scaled, labels))
    print(f"k = {k}   silhouette = {scores[-1]:.3f}")

plt.plot(range(2, 10), scores, marker="o")
plt.xlabel("k (number of clusters)")
plt.ylabel("average silhouette score")
plt.title("Silhouette score for each k")
plt.show()

**Question 3.** The silhouette score is highest at k = 2, but we have been working with k = 3. Should we switch?


**Answer**

*Provide your answer here*

### Case: segmenting cars

**In-class exercise.** You are working as a data analyst for a new auto consulting firm that helps car manufacturers position their vehicles more effectively in the market. The firm has collected data on car specifications, performance metrics, and prices from multiple brands. Your task is to use clustering to identify natural groupings of cars—such as economy, mid-range, and luxury segments—based on their technical and performance attributes. These clusters will help the marketing team understand how cars differ across brands and which features distinguish higher-end vehicles from budget-friendly models.

**Tips.**

- Choose the features for this task yourself, rather than using the full dataset. That is part of being an analyst!
- Should you include price?
- Break the task into small code cells. Run each one, review its output, and ask whether the step is really needed.

#### The cars dataset

| # | Feature | Description | Type |
|---|----------|--------------|------|
| 1 | Car_ID | Unique ID of each observation | Integer |
| 2 | Symboling | Assigned insurance risk rating; +3 = risky, -3 = safe | Categorical |
| 3 | carCompany | Name of the car company | Categorical |
| 4 | fueltype | Type of fuel used (gas or diesel) | Categorical |
| 5 | aspiration | Aspiration type used in the car | Categorical |
| 6 | doornumber | Number of doors in the car | Categorical |
| 7 | carbody | Body style of the car | Categorical |
| 8 | drivewheel | Type of drive wheel | Categorical |
| 9 | enginelocation | Location of the engine | Categorical |
| 10 | wheelbase | Wheelbase of the car | Numeric |
| 11 | carlength | Length of the car | Numeric |
| 12 | carwidth | Width of the car | Numeric |
| 13 | carheight | Height of the car | Numeric |
| 14 | curbweight | Weight of the car without occupants or baggage | Numeric |
| 15 | enginetype | Type of engine | Categorical |
| 16 | cylindernumber | Number of cylinders in the car | Categorical |
| 17 | enginesize | Engine size | Numeric |
| 18 | fuelsystem | Type of fuel system | Categorical |
| 19 | boreratio | Bore ratio of the engine | Numeric |
| 20 | stroke | Stroke or engine volume | Numeric |
| 21 | compressionratio | Compression ratio of the car | Numeric |
| 22 | horsepower | Horsepower | Numeric |
| 23 | peakrpm | Peak revolutions per minute (RPM) | Numeric |
| 24 | citympg | Mileage in the city | Numeric |
| 25 | highwaympg | Mileage on the highway | Numeric |
| 26 | price | Price of the car *(Dependent variable)* | Numeric |

#### Step 1 — load the data and look at it


In [ ]:
# Your code here -- load cars.csv into a DataFrame called cars and look at it

In [ ]:
# Your code here -- which columns are numeric, and are any values missing?

#### Step 2 — choose features

**Question 4.** The task asks for groups based on *technical and performance attributes*. Which columns belong in the clustering, and should `price` be one of them?

**Answer**

*Provide your answer here*

In [ ]:
# Your code here -- choose the clustering features

#### Step 3 — standardise, then choose *k*

Same trap as the bank data: `curbweight` runs into the thousands and `citympg` into the tens, so without standardising the clusters would just be "heavy, medium, light".


In [ ]:
# Your code here -- standardise the features and score a range of k

**Question 5.** Which *k* should the case use?

**Answer**

*Provide your answer here*

In [ ]:
# Your code here -- fit the segments and attach the labels to the cars table

#### Step 4 — profile the segments and name them

**Question 6.** Name the three segments. Do they differ on price, which the clustering never saw?

In [ ]:
# Your code here -- profile each segment, with price added back as a check

In [ ]:
# Your code here -- name the segments

In [ ]:
# Your code here -- cross-check the segments against columns you did not cluster on

In [ ]:
# Your code here -- plot two of the clustering variables, coloured by segment

**Answer**

*Provide your answer here*

**Exercise 2.** The code below is meant to reproduce the three car segments. Run it. It runs without error, but it produces a different, worse answer. Find the bug.

In [ ]:
# Fit three segments on the car specifications and profile them.
spec_features = ["wheelbase", "carlength", "carwidth", "curbweight",
                 "enginesize", "horsepower", "citympg", "highwaympg"]
cars_specs = cars[spec_features]

wrong_kmeans = KMeans(n_clusters=3, random_state=595, n_init=10)
cars["segment_wrong"] = wrong_kmeans.fit_predict(cars_specs)

cars.groupby("segment_wrong")[spec_features + ["price"]].mean().round(1)

**Answer**

*Provide your answer here*

### Follow up Questions and Exercises


**Question 1.** Supervised learning has a test set to tell you whether the model is any good. Clustering has no labels at all. So how do you know a clustering is right?


**Answer**

*Provide your answer here*

**Question 2.** Your manager asks for "the customer segments" and wants five of them, because the team has five campaign budgets. The silhouette scores say two. What do you do?


**Answer**

*Provide your answer here*

**Exercise 1.** Re-run the bank clustering using only the variables that describe the customer — age and balance — and leave out the campaign-mechanics columns (`day`, `duration`, `campaign`, `pdays`, `previous`). Does it produce a more useful segmentation?


In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## What comes next

Session 11 changes where the data comes from: instead of reading CSV files, you pull two tables out of a database with SQL, join them, and cluster the result with exactly today's recipe — numeric columns, standardise, fit, profile.

**Optional reading.** Linear regression for effect sizes, and statistical tests of the differences between clusters, are in `Optional/Supplemental_Statistics.ipynb`. That material is not examinable.